In [ ]:
# ----------------------------------------------------
# Title: Movie Gross Prediction Using SGD Regressor
# Author: Ehsan Saleh
# Course: Supervised Machine Learning
# Instructor: Dr. Ehsan Nazerfard
# Program: MCI AI Bootcamp (Generative AI with a Focus on Image Processing and Analysis)
# ----------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from google.colab import drive

# 1. Mount Google Drive to Google Colab
drive.mount('/content/drive')

# Your exact bootcamp file path
file_path = '/content/drive/MyDrive/MCI-AI-Bootcamp/Supervised ML/Question 1/CSM_dataset.xlsx'

# Load the dataset from Excel
df = pd.read_excel(file_path)

# 2. Define candidate features and target
candidate_features = ['Ratings', 'Budget', 'Screens', 'Sentiment', 'Views', 'Likes', 'Dislikes', 'Comments', 'Aggregate Followers']
target = 'Gross'

# Temporary dataframe to compute correlation (impute temporarily to avoid NaN issues in correlation)
df_temp = df[candidate_features + [target]].copy()
for col in df_temp.columns:
    df_temp[col] = df_temp[col].fillna(df_temp[col].median())

# Compute correlation matrix
correlation_matrix = df_temp.corr()

# Filter features based on correlation threshold (e.g., absolute correlation > 0.2)
threshold = 0.2
features_to_keep = []

print("—"*40)
print("Feature Correlation with Gross:")
print("—"*40)
for feature in candidate_features:
    corr_value = correlation_matrix.loc[feature, target]
    print(f"{feature}: {corr_value:.4f}")
    if abs(corr_value) >= threshold:
        features_to_keep.append(feature)

print("\n" + "—"*40)
print(f"Selected Features (Threshold >= {threshold}): {features_to_keep}")
print("—"*40)

# Create the final filtered dataframe based on automated selection
df_filtered = df[['Movie'] + features_to_keep + [target]].copy()

# 3. Impute missing values using Median for selected features
for col in features_to_keep + [target]:
    if df_filtered[col].isnull().sum() > 0:
        df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

# Save the filtered and imputed dataset to Google Drive
df_filtered.to_csv('/content/drive/MyDrive/MCI-AI-Bootcamp/Supervised ML/Question 1/CSM_dataset_filtered_imputed.csv', index=False)
print("\nFiltered and imputed dataset successfully saved.")

# 4. Separate features (X) and target (y) for normalization
X = df_filtered[features_to_keep].values
y = df_filtered[target].values

# Normalize features and target using StandardScaler
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# Save the normalized dataset to Google Drive
df_normalized = pd.DataFrame(X_scaled, columns=features_to_keep)
df_normalized['Gross_scaled'] = y_scaled
df_normalized.to_csv('/content/drive/MyDrive/MCI-AI-Bootcamp/Supervised ML/Question 1/CSM_dataset_normalized.csv', index=False)
print("Normalized dataset successfully saved.")

# 5. Initialize 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
rmse_scores = []

print("\n" + "—"*40)
print("Starting 5-Fold Cross Validation...")
print("—"*40)

# Iterate through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X_scaled), 1):
    # Split data into train and test sets for the current fold
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y_scaled[train_index], y_scaled[test_index]

    # 6. Train Linear Regression using Stochastic Gradient Descent (SGD)
    sgd = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
    sgd.fit(X_train, y_train)

    # Predict values for the current test fold
    y_pred_scaled = sgd.predict(X_test)

    # 7. Inverse transform predictions and actual values back to the original dollar scale
    y_test_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_true = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Calculate evaluation metrics for the current fold
    fold_mse = mean_squared_error(y_test_true, y_pred_true)
    fold_rmse = np.sqrt(fold_mse)

    mse_scores.append(fold_mse)
    rmse_scores.append(fold_rmse)

    print(f"Fold {fold} -> MSE: {fold_mse:,.2f} | RMSE: {fold_rmse:,.2f} USD")

# 8. Calculate and print overall average metrics across all 5 folds
print("\n" + "="*40)
print(f"Overall Average MSE: {np.mean(mse_scores):,.2f}")
print(f"Overall Average RMSE: {np.mean(rmse_scores):,.2f} USD")
print("="*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
————————————————————————————————————————
Feature Correlation with Gross:
————————————————————————————————————————
Ratings: 0.3422
Budget: 0.7200
Screens: 0.5660
Sentiment: -0.0171
Views: 0.1764
Likes: 0.1104
Dislikes: 0.1615
Comments: 0.1260
Aggregate Followers: 0.3135

————————————————————————————————————————
Selected Features (Threshold >= 0.2): ['Ratings', 'Budget', 'Screens', 'Aggregate Followers']
————————————————————————————————————————

Filtered and imputed dataset successfully saved.
Normalized dataset successfully saved.

————————————————————————————————————————
Starting 5-Fold Cross Validation...
————————————————————————————————————————
Fold 1 -> MSE: 1,383,106,182,788,803.00 | RMSE: 37,190,135.56 USD
Fold 2 -> MSE: 2,686,526,737,243,313.50 | RMSE: 51,831,715.55 USD
Fold 3 -> MSE: 2,745,553,533,233,191.00 | RMSE: 52,398,029.86 USD
Fold 4 -> MSE: 7,8